# Individual Case Study
MGSC 672: Operations and Supply Chain Analytics<br>
Emery Dittmer - 260658030 <br>
June 8<sup>th</sup> 2023

### Table of Contents
<div style= "background:white">
<ol> 
    <li><p><i><u><font size = '3'>Loading Data</font></u></i></p></li>
    <li><p><i><u><font size = '3'>Data Pre-processing</font></u></i></p></li>
    <li><p><i><u><font size = '3'>Demand Prediction</font></u></i></p></li>
    <li><p><i><u><font size = '3'>Forecasting LEC 24</font></u></i></p></li>
    </ol>
    

Note: this python scroipt produces slight different predictions that that of the 

## <font color=#FF0000>Step 1:Loading Data</font>

In [95]:
import pandas as pd
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.linear_model import LinearRegression
import numpy as np

In [96]:
df=pd.read_excel("Sourcing case students excel.xlsx",sheet_name="TV Daily Sales Data",skiprows=6,usecols=['Date','LEC ID','Units Sold'])
df=df.iloc[0:418,]
df.head()

,Date,LEC ID,Units Sold
0,2013-01-08 00:00:00,1.0,40.0
1,2013-01-09 00:00:00,1.0,5.0
2,2013-01-10 00:00:00,1.0,14.0
3,2013-01-11 00:00:00,1.0,8.0
4,2013-01-12 00:00:00,1.0,4.0


## <font color=#FF0000>Step 2:Data Pre-Processing</font>

### 2.1 General transformations

In [97]:
#get index column
#will need this for trend calculation
df=df.reset_index()

In [98]:
# Convert 'Date' column to datetime type
df['Date'] = pd.to_datetime(df['Date'])

In [99]:
# Sort the dataframe by 'Date' within each 'LEC ID' group
df.sort_values(['LEC ID', 'Date'], inplace=True)

### 2.2 Set period to number of days within each sale

In [100]:
# Calculate the number of days from the first day to the last day for each 'LEC ID'
df['time'] = df.groupby('LEC ID').cumcount() + 1
df.head(2)

,index,Date,LEC ID,Units Sold,time
0,0,2013-01-08,1.0,40.0,1
1,1,2013-01-09,1.0,5.0,2


### 2.3 Data Discovery

In [101]:
#check max LEC number of days
periods=df.groupby('LEC ID').agg({'time':'max'})
#Average number of periods
period=int(round(periods.mean(),0))
periods.head(2)

,time
LEC ID,
1.0,21
2.0,20


In [102]:
#understand the sales per LEC
inv=df.groupby('LEC ID').agg({'Units Sold':['sum','mean']})
inv

Units Sold           
              sum       mean
LEC ID                      
1.0         173.0   8.238095
2.0         196.0   9.800000
3.0         195.0  15.000000
4.0         210.0  14.000000
5.0         291.0  22.384615
6.0         170.0  12.142857
7.0         252.0  16.800000
8.0         520.0  24.761905
9.0         164.0  11.714286
10.0        135.0   9.642857
11.0        153.0  10.928571
12.0        322.0  20.125000
13.0        298.0  11.461538
14.0        133.0   8.312500
15.0        192.0  10.666667
16.0        198.0  14.142857
17.0        991.0  27.527778
18.0       1244.0  35.542857
19.0        855.0  30.535714
20.0        291.0  17.117647
21.0        143.0  13.000000
22.0        253.0  18.071429
23.0        385.0  29.615385

In [103]:
#average sales for any lec
inv['Units Sold']['sum'].mean()

337.5652173913044

In [104]:
#median sales for any lec
inv['Units Sold']['sum'].median()

210.0

In [105]:
inv.describe()

Units Sold           
               sum       mean
count    23.000000  23.000000
mean    337.565217  17.023155
std     294.639969   7.838670
min     133.000000   8.238095
25%     171.500000  11.195055
50%     210.000000  14.142857
75%     310.000000  21.254808
max    1244.000000  35.542857

## <font color=#FF0000>Step 3:Demand Prediction</font>

### De-seasonalise Demand

In [106]:
# Perform seasonal decomposition
result = seasonal_decompose(df['Units Sold'], model='additive', period=period)

# Retrieve the trend, seasonal, and residual components
trend = result.trend
seasonal = result.seasonal
residual = result.resid

# Print the de-seasonalized data
de_seasonalized = df['Units Sold'] - seasonal
df['de_seasonalized']=de_seasonalized
df.head()

,index,Date,LEC ID,Units Sold,time,de_seasonalized
0,0,2013-01-08,1.0,40.0,1,31.104325
1,1,2013-01-09,1.0,5.0,2,2.512154
2,2,2013-01-10,1.0,14.0,3,13.826547
3,3,2013-01-11,1.0,8.0,4,1.856851
4,4,2013-01-12,1.0,4.0,5,7.598012


In [107]:
# Reset index
df.reset_index(drop=True, inplace=True)
df['index']=df['index']+1

# Lambda function for a period of 18 days p=18 
sum_lambda = lambda row: (2*df.loc[max(row - 8, 0):min(row + 8, len(df) - 1), 'Units Sold'].sum()+ df.loc[row-9, 'Units Sold'] + df.loc[row+9, 'Units Sold'])/(2*18) if row >= 9 and row + 9 < len(df) else None

# Apply the lambda function to each row
df['de_seasonalized'] = df.index.map(sum_lambda)


In [108]:
df_clean=df.dropna()
X=np.array(df_clean['index']).reshape((-1, 1))
y=np.array(df_clean['de_seasonalized'])

# get regression
reg = LinearRegression().fit(X,y)

#get intercept and slope
intercept=reg.intercept_
slope=reg.coef_[0]

#add the demand to the dataframe
df['demand']=df['index']*slope+intercept

#add seasonal factor
df['seasonal factor']=df['Units Sold']/df['demand']
df


,index,Date,LEC ID,Units Sold,time,de_seasonalized,demand,seasonal factor
0,1,2013-01-08,1.0,40.0,1,NaN,10.458011,3.824819
1,2,2013-01-09,1.0,5.0,2,NaN,10.496478,0.476350
2,3,2013-01-10,1.0,14.0,3,NaN,10.534945,1.328911
3,4,2013-01-11,1.0,8.0,4,NaN,10.573412,0.756615
4,5,2013-01-12,1.0,4.0,5,NaN,10.611879,0.376936
...,...,...,...,...,...,...,...,...
413,414,2014-02-26,23.0,13.0,9,NaN,26.344926,0.493454
414,415,2014-02-27,23.0,23.0,10,NaN,26.383393,0.871760
415,416,2014-02-28,23.0,56.0,11,NaN,26.421860,2.119457
416,417,2014-03-01,23.0,37.0,12,NaN,26.460328,1.398320


In [109]:
#df['time']=[i % 18 for i in range(0,len(df))]
#df['time']=df['time']+1
df

,index,Date,LEC ID,Units Sold,time,de_seasonalized,demand,seasonal factor
0,1,2013-01-08,1.0,40.0,1,NaN,10.458011,3.824819
1,2,2013-01-09,1.0,5.0,2,NaN,10.496478,0.476350
2,3,2013-01-10,1.0,14.0,3,NaN,10.534945,1.328911
3,4,2013-01-11,1.0,8.0,4,NaN,10.573412,0.756615
4,5,2013-01-12,1.0,4.0,5,NaN,10.611879,0.376936
...,...,...,...,...,...,...,...,...
413,414,2014-02-26,23.0,13.0,9,NaN,26.344926,0.493454
414,415,2014-02-27,23.0,23.0,10,NaN,26.383393,0.871760
415,416,2014-02-28,23.0,56.0,11,NaN,26.421860,2.119457
416,417,2014-03-01,23.0,37.0,12,NaN,26.460328,1.398320


### Get seasonal demand

In [110]:
df_seas=df.groupby('time').agg({'seasonal factor':'mean'})
df_seas=df_seas.reset_index()

In [111]:
df_seas

,time,seasonal factor
0,1,2.103573
1,2,1.367689
2,3,0.927517
3,4,1.198328
4,5,0.733076
5,6,0.817703
6,7,0.749260
7,8,1.042806
8,9,0.762629
9,10,0.811327


### Add LEC 24 Data

In [112]:
#number of days to extend data from by
extend=21

LEC=[24] * extend
time=[i for i in range(1,extend+1)]
index=[i for i in range(int(df['index'].max()),int(df['index'].max())+extend)]

df1=pd.DataFrame(
    {
        'LEC ID':LEC,
        'time':time,
        'index':index
    }
)
df1['demand']=df1['index']*slope+intercept
df=pd.concat([df,df1])

### Get seasonal factors

In [113]:
df=df.merge(df_seas,how='left',left_on='time', right_on='time')
df

,index,Date,LEC ID,Units Sold,time,de_seasonalized,demand,seasonal factor_x,seasonal factor_y
0,1,2013-01-08,1.0,40.0,1,NaN,10.458011,3.824819,2.103573
1,2,2013-01-09,1.0,5.0,2,NaN,10.496478,0.476350,1.367689
2,3,2013-01-10,1.0,14.0,3,NaN,10.534945,1.328911,0.927517
3,4,2013-01-11,1.0,8.0,4,NaN,10.573412,0.756615,1.198328
4,5,2013-01-12,1.0,4.0,5,NaN,10.611879,0.376936,0.733076
...,...,...,...,...,...,...,...,...,...
434,434,NaT,24.0,NaN,17,NaN,27.114268,NaN,1.420661
435,435,NaT,24.0,NaN,18,NaN,27.152736,NaN,0.793561
436,436,NaT,24.0,NaN,19,NaN,27.191203,NaN,0.770067
437,437,NaT,24.0,NaN,20,NaN,27.229670,NaN,0.533420


In [114]:
df['seasonal factor']=df['seasonal factor_y']
df=df.drop(['seasonal factor_x','seasonal factor_y'],axis=1)
df['forecast']=df['demand']*df['seasonal factor']
df

,index,Date,LEC ID,Units Sold,time,de_seasonalized,demand,seasonal factor,forecast
0,1,2013-01-08,1.0,40.0,1,NaN,10.458011,2.103573,21.999188
1,2,2013-01-09,1.0,5.0,2,NaN,10.496478,1.367689,14.355920
2,3,2013-01-10,1.0,14.0,3,NaN,10.534945,0.927517,9.771337
3,4,2013-01-11,1.0,8.0,4,NaN,10.573412,1.198328,12.670417
4,5,2013-01-12,1.0,4.0,5,NaN,10.611879,0.733076,7.779312
...,...,...,...,...,...,...,...,...,...
434,434,NaT,24.0,NaN,17,NaN,27.114268,1.420661,38.520195
435,435,NaT,24.0,NaN,18,NaN,27.152736,0.793561,21.547340
436,436,NaT,24.0,NaN,19,NaN,27.191203,0.770067,20.939061
437,437,NaT,24.0,NaN,20,NaN,27.229670,0.533420,14.524860


In [115]:
#forecasting quantity for future demand for LEC 24
df[df['LEC ID']==24]['forecast'].sum()

514.0974693266334

In [116]:
df[df['LEC ID']==24]

,index,Date,LEC ID,Units Sold,time,de_seasonalized,demand,seasonal factor,forecast
418,418,NaT,24.0,NaN,1,NaN,26.498795,2.103573,55.742146
419,419,NaT,24.0,NaN,2,NaN,26.537262,1.367689,36.294727
420,420,NaT,24.0,NaN,3,NaN,26.575729,0.927517,24.649430
421,421,NaT,24.0,NaN,4,NaN,26.614196,1.198328,31.892538
422,422,NaT,24.0,NaN,5,NaN,26.652663,0.733076,19.538421
423,423,NaT,24.0,NaN,6,NaN,26.691130,0.817703,21.825417
424,424,NaT,24.0,NaN,7,NaN,26.729597,0.749260,20.027422
425,425,NaT,24.0,NaN,8,NaN,26.768064,1.042806,27.913899
426,426,NaT,24.0,NaN,9,NaN,26.806532,0.762629,20.443442
427,427,NaT,24.0,NaN,10,NaN,26.844999,0.811327,21.780062


In [117]:
#
df.to_csv('All data with forecast.csv')